# Bank Customer Churn Prediction using Neural Network

**Target:** `Exited`  
`0 = Customer stays`  
`1 = Customer leaves`

This notebook preprocesses the Bank Customer Churn dataset, encodes categorical variables, scales numerical features, trains a neural network classifier, evaluates it using accuracy, precision, recall, F1-score and confusion matrix, and predicts churn for a new customer.

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn tensorflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

print("TensorFlow version:", tf.__version__)


In [ ]:
# Upload Churn_Modelling.csv from Kaggle
uploaded = files.upload()


In [ ]:
# Load dataset
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
display(df.head())


## 1. Data Preprocessing

In [ ]:
# Check dataset information
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())


In [ ]:
# Remove columns that are identifiers and not useful predictive features
# RowNumber, CustomerId and Surname do not represent useful model features.
drop_columns = ["RowNumber", "CustomerId", "Surname"]

df = df.drop(columns=drop_columns, errors="ignore")

print("Columns after removing identifiers:")
print(df.columns.tolist())


In [ ]:
# Separate features and target
X = df.drop("Exited", axis=1)
y = df["Exited"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
# One-hot encode categorical variables
# drop_first=True avoids redundant dummy variables.
X = pd.get_dummies(
    X,
    columns=["Geography", "Gender"],
    drop_first=True
)

# Make sure all columns are numeric
X = X.astype(float)

print("Encoded feature shape:", X.shape)
display(X.head())


## 2. Explore Class Distribution

In [ ]:
print("Class distribution:")
print(y.value_counts())

print("\nClass percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(6,4))
sns.countplot(x=y)
plt.xticks([0,1], ["Stayed", "Exited"])
plt.title("Customer Churn Distribution")
plt.xlabel("Exited")
plt.ylabel("Number of Customers")
plt.show()


## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)


## 4. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")


## 5. Build Neural Network

In [ ]:
model = Sequential([
    Dense(32, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.20),

    Dense(16, activation="relu"),
    Dropout(0.20),

    Dense(8, activation="relu"),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 6. Train the Neural Network

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


## 7. Plot Training Performance

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()


## 8. Evaluate the Model

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("Test Loss     :", round(test_loss, 4))
print("Test Accuracy :", round(test_accuracy, 4))


In [ ]:
# Generate predictions
y_prob = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Stayed", "Exited"]
))


## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Stayed", "Exited"],
    yticklabels=["Stayed", "Exited"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


## 10. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7,5))
plt.plot(
    fpr,
    tpr,
    label=f"Neural Network (AUC = {roc_auc:.3f})"
)
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


## 11. Predict Churn for a New Customer

In [ ]:
# Example customer
# The values can be changed for a different customer.

new_customer = pd.DataFrame({
    "CreditScore": [650],
    "Geography": ["France"],
    "Gender": ["Female"],
    "Age": [40],
    "Tenure": [5],
    "Balance": [100000],
    "NumOfProducts": [2],
    "HasCrCard": [1],
    "IsActiveMember": [1],
    "EstimatedSalary": [60000]
})

# Apply the same one-hot encoding
new_customer = pd.get_dummies(
    new_customer,
    columns=["Geography", "Gender"],
    drop_first=True
)

# Ensure exactly the same feature columns as training data
new_customer = new_customer.reindex(
    columns=X.columns,
    fill_value=0
)

# Scale using the scaler fitted on training data
new_customer_scaled = scaler.transform(new_customer)

# Predict probability
churn_probability = model.predict(
    new_customer_scaled,
    verbose=0
)[0][0]

prediction = int(churn_probability >= 0.5)

print("Churn Probability:", round(churn_probability * 100, 2), "%")

if prediction == 1:
    print("Prediction: Customer is likely to EXIT.")
else:
    print("Prediction: Customer is likely to STAY.")


## 12. Save the Model

In [ ]:
# Save trained neural network and scaler
model.save("bank_customer_churn_model.keras")

import joblib
joblib.dump(scaler, "bank_customer_churn_scaler.pkl")

print("Model and scaler saved successfully.")


## Conclusion

A feed-forward neural network with ReLU hidden layers and a sigmoid output layer was trained to perform binary classification of bank customer churn. The model is evaluated using Accuracy, Precision, Recall, F1-score, ROC-AUC and a confusion matrix.